<a href="https://colab.research.google.com/github/BerkKilincer-dev/SoftIto/blob/main/Nlp/bpe_alt_kelime_tokenizasyon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tokenizasyon, Alt-Kelime (Subword) Parçalama ve BPE

Bir dil modeli metni doğrudan göremez; önce metni **token** denen parçalara ayırıp her birine bir sayı (ID) vermemiz gerekir. Bu not defterinde:

1. Kelime, karakter ve alt-kelime tokenizasyonunu karşılaştırıyoruz
2. **Byte Pair Encoding (BPE)** algoritmasını sıfırdan yazıyoruz
3. Byte-level BPE'nin (GPT-2, GPT-4, LLaMA-3) neden hiç `[UNK]` üretmediğini görüyoruz
4. Hugging Face `tokenizers` kütüphanesiyle aynı işi saniyeler içinde yapıyoruz
5. Gerçek LLM tokenizer'larında Türkçenin "token vergisi"ni ölçüyoruz

In [ ]:
import re
from collections import Counter
import random
import matplotlib.pyplot as plt
import pandas as pd

random.seed(42)
plt.rcParams["figure.dpi"] = 100

---
## 1) Neden alt-kelime?

| Yöntem | Örnek: `evlerimizden` | Avantaj | Dezavantaj |
|---|---|---|---|
| **Kelime** | `[evlerimizden]` | Kısa diziler | Devasa sözlük, sözlük dışı (OOV) kelime → `[UNK]` |
| **Karakter** | `[e, v, l, e, r, i, m, i, z, d, e, n]` | Küçük sözlük, OOV yok | Çok uzun diziler, anlam birimi yok |
| **Alt-kelime** | `[ev, ler, imiz, den]` | Orta boy sözlük, OOV yok, ekler paylaşılır | Tokenizer eğitmek gerekir |

Türkçe gibi **sondan eklemeli** dillerde tek bir kökten yüzlerce kelime türeyebilir. Kelime seviyesinde her biri ayrı bir sözlük girdisi olur; alt-kelimede ise `ev` + `ler` + `imiz` + `den` parçaları diğer kelimelerle **paylaşılır**.

### Eğitim korpusu

Küçük ama ek bakımından zengin bir Türkçe korpus üretiyoruz: kökler + büyük ünlü uyumuna göre seçilmiş ekler.

In [ ]:
INCE_UNLULER = set("eiöü")

def son_unlu(kelime):
    for harf in reversed(kelime):
        if harf in "aeıioöuü":
            return harf
    return "e"

def ek_al(kok, ince, kalin):
    return ince if son_unlu(kok) in INCE_UNLULER else kalin

def tureyen_kelime(kok):
    kelime = kok
    if random.random() < 0.6:
        kelime += ek_al(kelime, "ler", "lar")
    if random.random() < 0.5:
        kelime += ek_al(kelime, "imiz", "ımız")
    if random.random() < 0.6:
        kelime += random.choice([ek_al(kelime, "de", "da"), ek_al(kelime, "den", "dan"),
                                 ek_al(kelime, "e", "a"), ek_al(kelime, "in", "ın")])
    return kelime

kokler = ["ev", "okul", "kitap", "araba", "kalem", "masa", "bahçe", "şehir", "öğrenci",
          "öğretmen", "defter", "pencere", "kapı", "yol", "deniz", "orman", "köy", "bilgisayar"]
fiiller = ["gördük", "geldik", "baktık", "konuştuk", "çalıştık", "okuduk", "yürüdük", "bekledik"]
sifatlar = ["güzel", "büyük", "küçük", "eski", "yeni", "temiz", "sessiz", "kalabalık"]

cumleler = []
for _ in range(3000):
    kalip = random.choice([
        "{s} {k1} {f}",
        "{k1} {k2} {f}",
        "bugün {k1} çok {s}",
        "dün {s} {k1} {k2} {f}",
    ])
    cumleler.append(kalip.format(s=random.choice(sifatlar), f=random.choice(fiiller),
                                 k1=tureyen_kelime(random.choice(kokler)),
                                 k2=tureyen_kelime(random.choice(kokler))))

korpus = "\n".join(cumleler)
kelimeler = korpus.split()

print("Örnek cümleler:")
for c in cumleler[:6]:
    print("  •", c)
print(f"\nToplam kelime          : {len(kelimeler):,}")
print(f"Farklı kelime (sözlük) : {len(set(kelimeler)):,}")
print(f"Farklı karakter        : {len(set(korpus.replace(chr(10), '')))}")

---
## 2) BPE algoritması (Sennrich vd., 2016)

**Eğitim:**
1. Her kelimeyi karakterlerine ayır, sonuna kelime sonu işareti `</w>` ekle.
2. Korpustaki tüm **bitişik token çiftlerini** say.
3. En sık geçen çifti tek bir token olarak **birleştir** (merge) ve bu kuralı kaydet.
4. İstenen sözlük boyutuna ulaşana kadar 2–3'ü tekrarla.

**Kodlama (encode):** Yeni bir kelimeyi karakterlerine ayır ve öğrenilen birleştirme kurallarını **öğrenildikleri sırayla** uygula.

```
e v l e r </w>   →  ("e","r") birleşir  →  e v l er </w>
                 →  ("l","er")          →  e v ler </w>
                 →  ("e","v")           →  ev ler </w>
```

In [ ]:
def kelime_frekanslari(metin):
    """Her kelimeyi karakter tuple'ı + '</w>' olarak temsil eder: {('e','v','</w>'): 120, ...}"""
    sayac = Counter(metin.split())
    return {tuple(kelime) + ("</w>",): frekans for kelime, frekans in sayac.items()}


def cift_sayilari(sozluk):
    """Tüm bitişik token çiftlerini kelime frekansıyla ağırlıklandırarak sayar."""
    ciftler = Counter()
    for tokenlar, frekans in sozluk.items():
        for i in range(len(tokenlar) - 1):
            ciftler[(tokenlar[i], tokenlar[i + 1])] += frekans
    return ciftler


def cifti_birlestir(cift, sozluk):
    """Sözlükteki her kelimede verilen çifti tek tokena dönüştürür."""
    a, b = cift
    yeni_sozluk = {}
    for tokenlar, frekans in sozluk.items():
        yeni, i = [], 0
        while i < len(tokenlar):
            if i < len(tokenlar) - 1 and tokenlar[i] == a and tokenlar[i + 1] == b:
                yeni.append(a + b)
                i += 2
            else:
                yeni.append(tokenlar[i])
                i += 1
        yeni_sozluk[tuple(yeni)] = frekans
    return yeni_sozluk


def bpe_egit(metin, birlesme_sayisi, ayrintili=False):
    sozluk = kelime_frekanslari(metin)
    birlesmeler = []
    for adim in range(birlesme_sayisi):
        ciftler = cift_sayilari(sozluk)
        if not ciftler:
            break
        en_sik = max(ciftler, key=ciftler.get)
        sozluk = cifti_birlestir(en_sik, sozluk)
        birlesmeler.append(en_sik)
        if ayrintili and adim < 25:
            print(f"{adim + 1:3d}. birleşme: {en_sik[0]!r:>10} + {en_sik[1]!r:<10} → {en_sik[0] + en_sik[1]!r:14} (frekans: {ciftler[en_sik]})")
    return birlesmeler, sozluk

In [ ]:
birlesmeler, son_sozluk = bpe_egit(korpus, birlesme_sayisi=200, ayrintili=True)

İlk birleşmelerin **ekler** (`ler`, `lar`, `de`, `da`...) ve **sık kökler** olduğuna dikkat edin. Algoritma hiçbir dilbilgisi kuralı bilmeden Türkçenin morfolojisini veriden keşfediyor.

In [ ]:
def bpe_kodla(kelime, birlesmeler):
    tokenlar = list(kelime) + ["</w>"]
    sira = {cift: i for i, cift in enumerate(birlesmeler)}
    while True:
        # o anki tokenlarda uygulanabilecek, en erken öğrenilmiş birleşmeyi bul
        adaylar = [(sira[(tokenlar[i], tokenlar[i + 1])], i)
                   for i in range(len(tokenlar) - 1) if (tokenlar[i], tokenlar[i + 1]) in sira]
        if not adaylar:
            break
        _, i = min(adaylar)
        tokenlar = tokenlar[:i] + [tokenlar[i] + tokenlar[i + 1]] + tokenlar[i + 2:]
    return tokenlar

test_kelimeleri = ["evlerimizden", "kitaplarımızda", "öğretmenlerimize",
                   "arabalarımızın", "kalemlerden",
                   # korpusta HİÇ geçmeyen kelimeler:
                   "masalarımızdan", "denizlerimizde", "kediler", "telefonlarımızdan"]

for kelime in test_kelimeleri:
    korpusta = "✓" if kelime in set(kelimeler) else "✗ (yeni kelime)"
    print(f"{kelime:20s} {korpusta:18s} → {bpe_kodla(kelime, birlesmeler)}")

Korpusta hiç görülmemiş `telefonlarımızdan` bile `[UNK]` olmadı: bilinmeyen kök `telefon` karakterlere/küçük parçalara bölündü, ekler (`larımız`, `dan`) ise tanıdık tokenlar olarak çıktı.

### Birleşme sayısı ↔ dizi uzunluğu dengesi

In [ ]:
ornek_kelimeler = random.sample(kelimeler, 2000)
satirlar = []
for n in [0, 10, 25, 50, 100, 200, 400, 800]:
    b, _ = bpe_egit(korpus, n)
    ortalama = sum(len(bpe_kodla(k, b)) for k in ornek_kelimeler) / len(ornek_kelimeler)
    tum_tokenlar = {t for k in set(kelimeler) for t in bpe_kodla(k, b)}
    satirlar.append({"birleşme": n, "sözlük boyutu": len(tum_tokenlar), "token / kelime": ortalama})

denge = pd.DataFrame(satirlar)
fig, ax1 = plt.subplots(figsize=(9, 4.5))
ax1.plot(denge["birleşme"], denge["token / kelime"], "o-", color="tab:blue")
ax1.set_xlabel("birleşme sayısı"); ax1.set_ylabel("ortalama token / kelime", color="tab:blue")
ax2 = ax1.twinx()
ax2.plot(denge["birleşme"], denge["sözlük boyutu"], "s--", color="tab:red")
ax2.set_ylabel("kullanılan sözlük boyutu", color="tab:red")
plt.title("Daha fazla birleşme → daha kısa diziler ama daha büyük sözlük")
plt.show()
denge

---
## 3) Byte-level BPE: `[UNK]` tokeninin sonu

Karakter seviyesinde başlayan BPE, eğitimde hiç görmediği bir karakterle (emoji, Çince, `ğ` olmayan bir korpusta `ğ`) karşılaşırsa yine `[UNK]` üretir. GPT-2 ve sonrası bunu **baytlarla** başlayarak çözer:

- Başlangıç sözlüğü = 256 olası bayt değeri → her metin UTF-8 baytlarına çevrilebildiği için **sözlük dışı hiçbir şey yoktur**.
- Türkçe karakterler UTF-8'de **2 bayt** tutar; bu yüzden İngilizce ağırlıklı eğitilmiş tokenizer'larda Türkçe metin daha fazla token'a bölünür.

In [ ]:
for metin in ["a", "ğ", "ş", "€", "🚀", "merhaba", "güneş"]:
    baytlar = list(metin.encode("utf-8"))
    print(f"{metin!r:10} → {len(baytlar)} bayt: {baytlar}")

---
## 4) Hugging Face `tokenizers` ile BPE eğitmek

Aynı algoritmanın Rust ile yazılmış, üretim kalitesinde hali. Milyonlarca cümleyi saniyeler içinde işler.

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

hf_tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
hf_tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
egitici = trainers.BpeTrainer(vocab_size=300, min_frequency=2,
                              special_tokens=["[UNK]", "[PAD]", "[BOS]", "[EOS]"])
hf_tokenizer.train_from_iterator(cumleler, trainer=egitici)

print("Sözlük boyutu:", hf_tokenizer.get_vocab_size())
for metin in ["öğretmenlerimizden kitaplarımızı aldık", "denizlerimizde sessiz arabalar"]:
    kodlama = hf_tokenizer.encode(metin)
    print(f"\n{metin}")
    print("  tokenlar:", kodlama.tokens)
    print("  ID'ler  :", kodlama.ids)

### Byte-level BPE versiyonu (GPT-2 tarzı)

In [ ]:
bayt_tokenizer = Tokenizer(models.BPE())
bayt_tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
bayt_tokenizer.decoder = decoders.ByteLevel()
bayt_egitici = trainers.BpeTrainer(vocab_size=500, initial_alphabet=pre_tokenizers.ByteLevel.alphabet())
bayt_tokenizer.train_from_iterator(cumleler, trainer=bayt_egitici)

metin = "öğretmenlerimiz 🚀 ile 東京'a gitti"
kodlama = bayt_tokenizer.encode(metin)
print("Tokenlar :", kodlama.tokens)
print("Geri çöz :", bayt_tokenizer.decode(kodlama.ids))
print("\n'Ġ' = boşluk karakterinin bayt-seviyesi gösterimi. Emoji ve Japonca bile [UNK] olmadan kodlandı.")

---
## 5) Gerçek LLM tokenizer'ları ve Türkçenin "token vergisi"

Aynı anlamdaki cümle farklı dillerde farklı sayıda token tutar. Daha fazla token = **daha yüksek API maliyeti**, daha kısa etkin bağlam penceresi ve daha yavaş üretim.

> Bu hücre tokenizer dosyalarını Hugging Face Hub'dan indirir (birkaç MB).

In [ ]:
from transformers import AutoTokenizer

cumle_ciftleri = [
    ("Öğretmenlerimizden aldığımız kitapları okuduk.", "We read the books we got from our teachers."),
    ("Yarın sabah erkenden İstanbul'a gideceğiz.", "We will go to Istanbul early tomorrow morning."),
    ("Bu yapay zekâ modeli Türkçe metinleri anlayabiliyor mu?", "Can this AI model understand Turkish texts?"),
]

tokenizer_adlari = {
    "GPT-2 (2019, İngilizce ağırlıklı)": "openai-community/gpt2",
    "Qwen2.5 (çok dilli)": "Qwen/Qwen2.5-0.5B-Instruct",
    "BERTurk (Türkçe WordPiece)": "dbmdz/bert-base-turkish-cased",
}

satirlar = []
for etiket, ad in tokenizer_adlari.items():
    tok = AutoTokenizer.from_pretrained(ad)
    tr = sum(len(tok.tokenize(t)) for t, _ in cumle_ciftleri)
    en = sum(len(tok.tokenize(e)) for _, e in cumle_ciftleri)
    satirlar.append({"Tokenizer": etiket, "Türkçe token": tr, "İngilizce token": en,
                     "TR / EN oranı": round(tr / en, 2)})
    print(f"{etiket}\n  → {tok.tokenize(cumle_ciftleri[0][0])}\n")

pd.DataFrame(satirlar)

---
## BPE ailesi: kısa karşılaştırma

| Algoritma | Kullanan modeller | Birleştirme kriteri |
|---|---|---|
| **BPE** | GPT-2/3/4, LLaMA-3, Qwen, Mistral | En **sık** çifti birleştir |
| **WordPiece** | BERT, BERTurk, DistilBERT | Olabilirliği en çok artıran çifti birleştir: $\frac{f(ab)}{f(a)\,f(b)}$ |
| **Unigram LM** (SentencePiece) | T5, ALBERT, XLNet, LLaMA-1/2 | Büyük sözlükten başla, kaybı en az artıran tokenları **sil** |

## Özet

- Alt-kelime tokenizasyon, kelime ve karakter seviyesinin en iyi yanlarını birleştirir.
- BPE en sık bitişik çiftleri tekrar tekrar birleştirerek sözlük oluşturur; Türkçe ekleri kendiliğinden yakalar.
- **Byte-level BPE** 256 baytla başladığı için hiçbir zaman `[UNK]` üretmez.
- Tokenizer'ın eğitildiği dil dağılımı, o dildeki maliyeti ve performansı doğrudan etkiler.